# Code used to generate canopy width data for allometric equations

Canopy data comes from: 
Pawlak CC, Yost JM, Ventura J, Guizan G, Arnold S, Okin GS, et al. Mapping California’s Urban Forest at Scale: An Error-Adjusted Canopy Time Series for Monitoring Change. Ecology; 2026. doi:10.64898/2026.05.04.722774 [PREPRINT]


In [7]:
# Canopy rasters are from: https://www.biorxiv.org/content/10.64898/2026.05.04.722774v1.full
# ZipCodes are from: https://www.census.gov/cgi-bin/geo/shapefiles/index.php (2020, CA zipcodes)

import os
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.features import shapes as rio_shapes
from shapely.geometry import shape
from tqdm import tqdm

# ==========================================================
# INPUTS
# ==========================================================
CSV_PATH = r"PATH TO WCA DATASET WITH POINT LOCATIONS"
ZIP_PATH = r"PATH\california_shps\ca_zip_equalArea.shp"
ZIP_COL = "zip_code"

OUT_DIR = Path(r"OUTPATH")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CRS = "EPSG:4326"   # or zip_gdf.crs if you want

RASTERS = {
    2020: {
        10: r"PATH\full_state_utm_10_2020_crop.tif",
        11: r"PATH\full_state_utm_11_2020_crop_v2.tif",
    },
    2022: {
        10: r"PATH\full_state_utm_10_2022_crop.tif",
        11: r"PATH\full_state_utm_11_2022_crop.tif",
    },
}

YEARS_RASTER = [2020, 2022]
MIN_POINTS_PER_ZIP = 10

LAT_COL = "latitude"
LON_COL = "longitude"
DATE_COL = "date_worked"

In [ ]:
# ==========================================================
# HELPERS
# ==========================================================
def normalize_zip(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    out = pd.Series(pd.NA, index=series.index, dtype="string")
    ok = s.notna()
    out.loc[ok] = s.loc[ok].astype(int).astype(str).str.zfill(5)
    return out

def _valid_xy_mask(df: pd.DataFrame) -> pd.Series:
    return (
        df[LAT_COL].notna()
        & df[LON_COL].notna()
        & (df[LAT_COL] != 0)
        & (df[LON_COL] != 0)
    )

def _bbox_intersects(bounds_a, bounds_b) -> bool:
    # (minx, miny, maxx, maxy)
    return not (
        bounds_a[2] < bounds_b[0]
        or bounds_a[0] > bounds_b[2]
        or bounds_a[3] < bounds_b[1]
        or bounds_a[1] > bounds_b[3]
    )

def _raster_bounds_tuple(ds) -> tuple:
    b = ds.bounds
    return (b.left, b.bottom, b.right, b.top)

def polygonize_raster_ones_in_zip(ds, zip_geom_ds_crs):
    """
      - windowed read using the ZIP geometry
      - polygonize ONLY where raster==1
      - return polygons in ds CRS
    """
    out_img, out_transform = rio_mask(
        ds,
        [zip_geom_ds_crs],
        crop=True,
        filled=True,
        nodata=0,          # treat outside/NoData as 0 (not canopy)
        all_touched=False,
    )
    arr = out_img[0]  # single-band assumed

    if arr.size == 0:
        return []

    mask_ones = (arr == 1)
    if not np.any(mask_ones):
        return []

    polys = []
    arr_u8 = arr.astype(np.uint8, copy=False)
    for geom, val in rio_shapes(arr_u8, mask=mask_ones, transform=out_transform):
        if int(val) == 1:
            polys.append(shape(geom))
    return polys

def polys_overlapping_points(polys, points_ds_crs: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Keep ONLY canopy polygons that overlap any of the POINTS (in the same CRS).
    Uses intersects (robust to edge cases). You can switch to "contains" if you want.
    """
    if len(polys) == 0 or points_ds_crs.empty:
        return gpd.GeoDataFrame(geometry=[], crs=points_ds_crs.crs)

    poly_gdf = gpd.GeoDataFrame(
        {"poly_id": np.arange(len(polys), dtype=int)},
        geometry=polys,
        crs=points_ds_crs.crs,
    )

    j = gpd.sjoin(poly_gdf, points_ds_crs[["geometry"]], how="inner", predicate="intersects")
    if j.empty:
        return poly_gdf.iloc[0:0].copy()

    keep_ids = j["poly_id"].unique()
    return poly_gdf.loc[poly_gdf["poly_id"].isin(keep_ids)].copy()

# ==========================================================
# 1) LOAD POINTS
# ==========================================================
df = pd.read_csv(CSV_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df["year"] = df[DATE_COL].dt.year

df = df[df["year"].isin(YEARS_RASTER)].copy()
df = df[_valid_xy_mask(df)].copy()

pts_wgs84 = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df[LON_COL], df[LAT_COL]),
    crs="EPSG:4326",
)

# ==========================================================
# 2) LOAD ZIP POLYGONS + JOIN POINTS TO ZIPS
# ==========================================================
zip_gdf = gpd.read_file(ZIP_PATH)
zip_gdf[ZIP_COL] = normalize_zip(zip_gdf[ZIP_COL])


if ZIP_COL not in zip_gdf.columns:
    raise ValueError(f"ZIP_COL='{ZIP_COL}' not found in ZIP shapefile columns: {list(zip_gdf.columns)}")

# join in ZIP CRS (so "within" is meaningful wrt zip polygons)
pts_in_zip = gpd.sjoin(
    pts_wgs84.to_crs(zip_gdf.crs),
    zip_gdf[[ZIP_COL, "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"], errors="ignore")

pts_in_zip[ZIP_COL] = normalize_zip(pts_in_zip[ZIP_COL])
pts_in_zip = pts_in_zip.dropna(subset=[ZIP_COL]).copy()


zip_counts = (
    pts_in_zip.groupby(["year", ZIP_COL], as_index=False)
    .size()
    .rename(columns={"size": "n_points"})
)

zip_keep = zip_counts[zip_counts["n_points"] > MIN_POINTS_PER_ZIP].copy()

print("\nZIPs per year with > 10 points:")
print(zip_keep.sort_values(["year", "n_points"], ascending=[True, False]).to_string(index=False))

# only keep zip geometries we actually need

zip_need = zip_gdf[zip_gdf[ZIP_COL].isin(zip_keep[ZIP_COL].unique())][[ZIP_COL, "geometry"]].copy()


zip_need[ZIP_COL] = zip_need[ZIP_COL].astype(str)

# ==========================================================
# 3) MAIN LOOP:
#    GIVEN (year, zip):
#      - take that zip geometry
#      - open raster(s) for that year (utm10, utm11)
#      - if zip intersects raster extent -> window read by zip geometry
#      - polygonize 1s
#      - keep polygons overlapping the POINTS for that (year, zip)
#      - append to output list
#    SAVE: one GPKG per year
# ==========================================================
for year in YEARS_RASTER:
    zips_year = zip_keep.loc[zip_keep["year"] == year, ZIP_COL].tolist()
    if len(zips_year) == 0:
        print(f"\n{year}: no ZIPs with > {MIN_POINTS_PER_ZIP} points. skipping.")
        continue

    out_gpkg = OUT_DIR / f"canopy_polys_overlap_points_{year}.gpkg"
    if out_gpkg.exists():
        out_gpkg.unlink()

    kept_all = []

    for zip_code in tqdm(zips_year, desc=f"{year} ZIPs"):
        # zip geometry (in zip CRS for now)
        zip_row = zip_need[zip_need[ZIP_COL] == str(zip_code)]
        if zip_row.empty:
            continue
        zip_geom_zipcrs = zip_row.geometry.iloc[0]

        # points for this (year, zip) in zip CRS (we will reproject later per raster)
        pts_yz_zipcrs = pts_in_zip[(pts_in_zip["year"] == year) & (pts_in_zip[ZIP_COL] == str(zip_code))].copy()
        if pts_yz_zipcrs.empty:
            continue

        # loop rasters ONE BY ONE (utm10 then utm11) as you requested
        for zone, rpath in RASTERS[year].items():
            if not os.path.exists(rpath):
                print(f"Missing raster: {rpath}")
                continue

            with rasterio.open(rpath) as ds:
                if ds.crs is None:
                    raise ValueError(f"Raster CRS is None: {rpath}")

                # reproject zip geometry AND points to THIS raster CRS
                zip_geom_ds = gpd.GeoSeries([zip_geom_zipcrs], crs=zip_gdf.crs).to_crs(ds.crs).iloc[0]
                pts_ds = pts_yz_zipcrs.to_crs(ds.crs)

                # extent check (so we skip non-overlapping zone)
                if not _bbox_intersects(zip_geom_ds.bounds, _raster_bounds_tuple(ds)):
                    continue

                # 1) window read exactly within zip geometry
                polys = polygonize_raster_ones_in_zip(ds, zip_geom_ds)
                if len(polys) == 0:
                    continue

                # 2) keep only polygons overlapping those points
                kept = polys_overlapping_points(polys, pts_ds)
                if kept.empty:
                    continue

                kept["year"] = int(year)
                kept["zip_code"] = str(zip_code)
                kept["utm_zone"] = int(zone)
                kept["raster_path"] = rpath
                kept = kept.to_crs(OUT_CRS)   

                kept_all.append(kept)

    if len(kept_all) == 0:
        print(f"\n{year}: no overlapping polygons found.")
        continue

    out_gdf = pd.concat(kept_all, ignore_index=True)
    out_gdf = gpd.GeoDataFrame(out_gdf, geometry="geometry", crs=OUT_CRS)

    out_gdf.to_file(out_gpkg, layer=f"overlap_{year}", driver="GPKG")
    print(f"\nSaved: {out_gpkg} | layer=overlap_{year} | n_polys={len(out_gdf)}")

    # quick sanity stats
    stats = (
        out_gdf.groupby(["utm_zone"], as_index=False)
        .size()
        .rename(columns={"size": "n_polys"})
    )
    print("\nPolygons kept by UTM zone:")
    print(stats.to_string(index=False))


In [ ]:
import pandas as pd
import geopandas as gpd
# Tree detector points are from: https://osf.io/4s859/overview

# J. Ventura, C. Pawlak, M. Honsberger, C. Gonsalves, J. Rice, N. Love, S. Han, V. Nguyen, 
# K. Sugano, J. Doremus, G.A. Fricker, J. Yost & M. Ritter (2024). 
# Results: Individual Tree Detection in Large-Scale Urban Environments using High-Resolution
# Multispectral Imagery [dataset]. Open Science Framework. DOI 10.17605/OSF.IO/4S859

# ==========================================================
# USER INPUT: detected tree points CSV
# ==========================================================
DETECTED_POINTS_CSV = r"PATH TO TREE POINTS"  # <-- change
PT_LAT_COL = "lat"   # <-- change
PT_LON_COL = "lon"   # <-- change

# ==========================================================
# LOAD detected points (WGS84) -> GeoDataFrame
# ==========================================================
pts = pd.read_csv(DETECTED_POINTS_CSV)

# basic sanity filter (optional)
pts = pts.dropna(subset=[PT_LAT_COL, PT_LON_COL]).copy()
pts = pts[(pts[PT_LAT_COL] != 0) & (pts[PT_LON_COL] != 0)].copy()

pts_gdf = gpd.GeoDataFrame(
    pts,
    geometry=gpd.points_from_xy(pts[PT_LON_COL], pts[PT_LAT_COL]),
    crs="EPSG:4326",
)

# ==========================================================
# ADD COUNTS TO EACH YEAR'S POLYGON OUTPUT
# ==========================================================
for year in YEARS_RASTER:
    out_gpkg = OUT_DIR / f"canopy_polys_overlap_points_{year}.gpkg"
    layer_name = f"overlap_{year}"

    if not out_gpkg.exists():
        print(f"Missing polygons for {year}: {out_gpkg}")
        continue

    polys = gpd.read_file(out_gpkg, layer=layer_name)

    # reproject points to polygon CRS
    pts_proj = pts_gdf.to_crs(polys.crs)

    # spatial join: attach polygon index to each point
    # predicate="within" is typical: point in polygon
    j = gpd.sjoin(
        pts_proj[["geometry"]],
        polys[["geometry"]],
        how="left",
        predicate="within",
    )

    # count points per polygon (index_right = polygon index)
    counts = (
        j["index_right"]
        .value_counts(dropna=True)
        .rename("num_detected_trees")
    )

    # default 0, then fill where counts exist
    polys["num_detected_trees"] = 0
    polys.loc[counts.index.astype(int), "num_detected_trees"] = counts.values

    # save to a NEW layer so you keep the original too (recommended)
    out_layer = f"{layer_name}_with_detected"
    

    from pathlib import Path

    out_gpkg = Path(out_gpkg)
    
    # delete old version completely
    if out_gpkg.exists():
        out_gpkg.unlink()
    
    # write new version with your updated method
    polys.to_file(out_gpkg, layer=out_layer, driver="GPKG")

    print(f"{year}: wrote {out_layer} with num_detected_trees to {out_gpkg}")


In [ ]:
import geopandas as gpd
import pandas as pd

# ==========================================================
# INPUTS
# ==========================================================
# uses: OUT_DIR, YEARS_RASTER from your script
# assumes you wrote layers named: overlap_{year}_with_detected
# and they include: num_detected_trees

layers = []
for year in YEARS_RASTER:
    gpkg = OUT_DIR / f"canopy_polys_overlap_points_{year}.gpkg"
    layer = f"overlap_{year}_with_detected"

    if not gpkg.exists():
        print(f"Missing: {gpkg}")
        continue

    g = gpd.read_file(gpkg, layer=layer)
    g["year"] = int(year)  # ensure present
    layers.append(g)

if len(layers) == 0:
    raise RuntimeError("No yearly polygon layers found to combine.")

# combine (CRS should already match; if not, harmonize to the first)
base_crs = layers[0].crs
layers = [gi.to_crs(base_crs) if gi.crs != base_crs else gi for gi in layers]

all_polys = gpd.GeoDataFrame(pd.concat(layers, ignore_index=True), crs=base_crs)

# subset
one_detected = all_polys[all_polys["num_detected_trees"].fillna(0).astype(int) == 1].copy()

print("Combined polygons:", len(all_polys))
print("num_detected_trees == 1:", len(one_detected))

# optional save
# out_gpkg = OUT_DIR / "canopy_polys_all_years.gpkg"
# all_polys.to_file(out_gpkg, layer="all_years_with_detected", driver="


In [ ]:
# get species of canopies

import pandas as pd
import geopandas as gpd

# The following cannot be run with clean data

CSV_PATH = r"PATH TO WCA DATASET WITH POINT LOCATIONS"
LAT_COL = "latitude"
LON_COL = "longitude"
DATE_COL = "date_worked"
YEARS_KEEP = None  # e.g. [2020, 2022]

# ---- points (keep ALL columns) ----
dfp = pd.read_csv(CSV_PATH)
dfp[DATE_COL] = pd.to_datetime(dfp[DATE_COL], errors="coerce")
dfp["year"] = dfp[DATE_COL].dt.year

mask_pts = (
    dfp[LAT_COL].notna()
    & dfp[LON_COL].notna()
    & (dfp[LAT_COL] != 0)
    & (dfp[LON_COL] != 0)
)
dfp = dfp.loc[mask_pts].copy()
if YEARS_KEEP is not None:
    dfp = dfp[dfp["year"].isin(YEARS_KEEP)].copy()

pts = gpd.GeoDataFrame(
    dfp,
    geometry=gpd.points_from_xy(dfp[LON_COL], dfp[LAT_COL]),
    crs="EPSG:4326",
)

# stable point id so we can count without index_right
pts = pts.reset_index(drop=True)
pts["pt_idx"] = pts.index.astype(int)

# ---- polygons (keep ALL columns) ----
polys = one_detected.copy().reset_index(drop=True)
polys["poly_idx"] = polys.index.astype(int)

# reproject points to polygon CRS
pts = pts.to_crs(polys.crs)

# ---- join with polygons on LEFT so geometry stays POLYGONS ----
j_poly = gpd.sjoin(
    polys,
    pts,
    how="left",
    predicate="contains",   # use "intersects" if you want boundary hits
    lsuffix="poly",
    rsuffix="pt",
)

# ---- count points per polygon using pt_idx (NOT index_right) ----
n_per_poly = (
    j_poly.groupby("poly_idx")["pt_idx"]
    .apply(lambda s: s.notna().sum())
    .astype(int)
)

j_poly["n_points_in_poly"] = j_poly["poly_idx"].map(n_per_poly).fillna(0).astype(int)

# ---- keep polygons with exactly 1 point and keep ALL joined columns ----
polys_with_point_cols = (
    j_poly[j_poly["n_points_in_poly"] == 1]
    .sort_values("poly_idx")
    .drop_duplicates(subset=["poly_idx"], keep="first")
    .copy()
)

polys_with_point_cols = gpd.GeoDataFrame(polys_with_point_cols, geometry="geometry", crs=polys.crs)

print("Polygons before:", len(polys))
print("Polygons kept (exactly 1 point):", len(polys_with_point_cols))
print("Geometry types:", polys_with_point_cols.geom_type.value_counts().to_dict())
print("Columns retained:", len(polys_with_point_cols.columns))


In [ ]:
import numpy as np

# ---- compute canopy widths from polygon bounds (in meters), then average ----
# Use an equal-area / metric CRS so widths + area are in meters/m^2
METRIC_CRS = "EPSG:3310"

# sanity: must be polygons as active geometry
if polys_with_point_cols.geom_type.isin(["Point", "MultiPoint"]).any():
    raise ValueError("polys_with_point_cols active geometry is points. Set polygon geometry as active first.")

g_metric = polys_with_point_cols.to_crs(METRIC_CRS)

# bounds-based canopy widths
minx, miny, maxx, maxy = g_metric.geometry.bounds.T.values

polys_with_point_cols["canopy_width_ew_m"] = (maxx - minx).astype(float)  # E-W
polys_with_point_cols["canopy_width_ns_m"] = (maxy - miny).astype(float)  # N-S
polys_with_point_cols["canopy_width_avg_m"] = (
    (polys_with_point_cols["canopy_width_ew_m"] + polys_with_point_cols["canopy_width_ns_m"]) / 2.0
).astype(float)

# canopy area (polygon area in m^2) + optional conversions
polys_with_point_cols["canopy_area_m2"] = g_metric.geometry.area.astype(float)
polys_with_point_cols["canopy_area_ft2"] = polys_with_point_cols["canopy_area_m2"] * 10.763910416709722


# optional: feet for widths
polys_with_point_cols["canopy_width_ew_ft"] = polys_with_point_cols["canopy_width_ew_m"] * 3.280839895
polys_with_point_cols["canopy_width_ns_ft"] = polys_with_point_cols["canopy_width_ns_m"] * 3.280839895
polys_with_point_cols["canopy_width_avg_ft"] = polys_with_point_cols["canopy_width_avg_m"] * 3.280839895

polys_with_point_cols[[
    "canopy_width_ns_m","canopy_width_ew_m","canopy_width_avg_m",
    "canopy_width_ns_ft","canopy_width_ew_ft","canopy_width_avg_ft",
    "canopy_area_m2","canopy_area_ft2"
]].describe()


In [ ]:
import numpy as np

# ==========================================================
# COMPUTE THINNESS / COMPACTNESS
# ==========================================================

METRIC_CRS = "EPSG:3310"

# sanity: active geometry should be polygons, not points
if polys_with_point_cols.geom_type.isin(["Point", "MultiPoint"]).any():
    raise ValueError(
        "polys_with_point_cols active geometry is points. "
        "Set polygon geometry as active first."
    )

# project to metric CRS for accurate area and perimeter
g_metric = polys_with_point_cols.to_crs(METRIC_CRS)

# area and perimeter
polys_with_point_cols["area_m2"] = g_metric.geometry.area.astype(float)
polys_with_point_cols["perim_m"] = g_metric.geometry.length.astype(float)

# thinness ratio / compactness
# 1.0 = perfect circle
# closer to 0 = thinner / more elongated / more jagged
polys_with_point_cols["thinness"] = (
    (4 * np.pi * polys_with_point_cols["area_m2"]) /
    (polys_with_point_cols["perim_m"] ** 2)
)

# avoid inf/nan issues for invalid or zero-perimeter geometries
polys_with_point_cols.loc[
    polys_with_point_cols["perim_m"] <= 0,
    "thinness"
] = np.nan

polys_with_point_cols[
    ["area_m2", "perim_m", "thinness"]
].describe()

In [ ]:
# ==========================================================
# KEEP ONLY POLYGONS WITH THINNESS > 0.5
# ==========================================================

polys_thinness_filtered = polys_with_point_cols[
    polys_with_point_cols["thinness"] > 0.5
].copy()

print(f"Original polygons: {len(polys_with_point_cols):,}")
print(f"Kept polygons with thinness > 0.5: {len(polys_thinness_filtered):,}")
print(f"Removed polygons: {len(polys_with_point_cols) - len(polys_thinness_filtered):,}")

polys_thinness_filtered[["area_m2", "perim_m", "thinness"]].describe()

In [ ]:
# ---------------------------------------------------------
# MAKE SURE WE ARE ONLY USING THINNESS-FILTERED POLYGONS
# ---------------------------------------------------------
if "thinness" not in polys_thinness_filtered.columns:
    raise ValueError("thinness column not found in polys_thinness_filtered.")

if (polys_thinness_filtered["thinness"] <= 0.5).any():
    raise ValueError("polys_thinness_filtered still contains polygons with thinness <= 0.5.")

print(f"Using thinness-filtered polygons only: {len(polys_thinness_filtered):,} rows")
print(polys_thinness_filtered["thinness"].describe())

In [ ]:
# --- rename + drop ---
out = polys_thinness_filtered.copy()

# rename year_poly -> canopy_year if present
if "year_poly" in out.columns:
    out = out.rename(columns={"year_poly": "canopy_year"})

drop_cols = [
    "geometry",
    "num_detected_trees",
    "utm_zone",
    "raster_path",
    "zip_code",
    "year_pt",
    "pt_idx",
    "n_points_in_poly",
]

out = out.drop(columns=[c for c in drop_cols if c in out.columns])

# --- save ---
out_csv = r"PATH\canopy_data.csv"
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
out.to_csv(out_csv, index=False)

print("Saved:", out_csv)
print("Columns:", list(out.columns))
print("Rows:", len(out))
print("Minimum thinness saved:", out["thinness"].min())